# Chicago TNP Trips — Daily Demand Forecast

Predict **daily trip demand per area (H3 cell)** for the busiest cells, using calendar
and past-demand features. A linear baseline is compared against LightGBM.

Data source: MatrixOne (`chicago_tnp.trips`), Chicago rideshare trips, 100k/month sample.

In [ ]:
# Install dependencies (run once)
%pip install -q pandas numpy scikit-learn lightgbm pymysql

## 1. Build the target: daily demand per cell

Count trips per (cell, day). We use **daily** granularity (not hourly): the data is a
~1.5% sample, so hourly cells are mostly noise/zeros, while daily counts are stable.

In [ ]:
import pymysql
import pandas as pd

conn = pymysql.connect(host="127.0.0.1", port=6001,
                       user="root", password="111", database="chicago_tnp")

sql = """
SELECT pickup_h3, DATE(trip_start_timestamp) AS d, COUNT(*) AS demand
FROM trips
WHERE shared_trip_authorized = 0
GROUP BY pickup_h3, DATE(trip_start_timestamp)
"""
daily = pd.read_sql(sql, conn)
conn.close()

daily["d"] = pd.to_datetime(daily["d"])
print("rows:", len(daily))
daily.head()

## 2. Keep the busiest cells and fill missing days with 0

Demand is very concentrated: the top 50 cells hold ~54% of all trips and have enough
data to learn from. We keep those, then fill any missing (cell, day) with demand = 0.

A check (not shown here) confirmed these zeros are **real low-demand days**, scattered
across all months, not contiguous data gaps — so they are kept as true zeros.

In [ ]:
top_cells = (daily.groupby("pickup_h3")["demand"].sum()
             .sort_values(ascending=False).head(50).index.tolist())
daily = daily[daily["pickup_h3"].isin(top_cells)].copy()

# Full (cell x day) grid, then fill gaps with 0
days = pd.date_range(daily["d"].min(), daily["d"].max(), freq="D")
skeleton = pd.MultiIndex.from_product([top_cells, days],
                                      names=["pickup_h3", "d"]).to_frame(index=False)
full = skeleton.merge(daily, on=["pickup_h3", "d"], how="left")
full["demand"] = full["demand"].fillna(0).astype(int)

print("rows:", len(full), "| zero ratio:", round((full["demand"] == 0).mean() * 100, 1), "%")

## 3. Features

Give the model clues to predict demand: calendar, cell identity, and **past demand**
(lags + rolling averages). Past demand is the strongest signal for future demand.

In [ ]:
import numpy as np

full = full.sort_values(["pickup_h3", "d"]).reset_index(drop=True)
dt = full["d"].dt

# Calendar
full["dow"]         = dt.dayofweek            # 0=Mon ... 6=Sun
full["is_weekend"]  = (dt.dayofweek >= 5).astype("int8")
full["month"]       = dt.month
full["day_of_year"] = dt.dayofyear

# Cell identity (each area has a very different demand level)
full["cell_code"] = full["pickup_h3"].astype("category").cat.codes

# Lags: demand N days ago
g = full.groupby("pickup_h3")["demand"]
full["lag_1"]  = g.shift(1)
full["lag_7"]  = g.shift(7)
full["lag_14"] = g.shift(14)

# Rolling means. shift(1) FIRST so the window never includes today (avoids leakage).
full["roll_7"]  = g.shift(1).rolling(7).mean().reset_index(level=0, drop=True)
full["roll_28"] = g.shift(1).rolling(28).mean().reset_index(level=0, drop=True)

# Each cell's baseline level, computed on the TRAIN period only (avoids leakage)
cell_mean = (full[full["d"] < "2024-01-01"].groupby("pickup_h3")["demand"]
             .mean().rename("cell_mean"))
full = full.merge(cell_mean, on="pickup_h3", how="left")

feat_cols = ["dow", "is_weekend", "month", "day_of_year", "cell_code",
             "lag_1", "lag_7", "lag_14", "roll_7", "roll_28", "cell_mean"]
full = full.dropna(subset=feat_cols).reset_index(drop=True)   # drop early rows with no history
print("rows after features:", len(full))
full.head()

## 4. Train / test split — by time, not random

We predict the future, so we train on the past (2022-2023) and test on the unseen
future (2024). A **random** split would leak future info through the lag features
(tomorrow's row would carry today's demand as a feature into training).

In [ ]:
train = full[full["d"] < "2024-01-01"]
test  = full[full["d"] >= "2024-01-01"]

X_train, y_train = train[feat_cols], train["demand"]
X_test,  y_test  = test[feat_cols],  test["demand"]

print("train:", len(X_train), train["d"].min().date(), "->", train["d"].max().date())
print("test :", len(X_test),  test["d"].min().date(),  "->", test["d"].max().date())

## 5. Baseline: Linear Regression

Always build a simple baseline first, so a complex model's score has something to beat.
MAE = mean absolute error = average miss in trips. We also show it as a % of mean demand.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

lr = LinearRegression().fit(X_train, y_train)
pred_lr = np.clip(lr.predict(X_test), 0, None)        # demand can't be negative
mae_lr = mean_absolute_error(y_test, pred_lr)
print(f"Linear MAE: {mae_lr:.2f}  ({mae_lr / y_test.mean() * 100:.1f}% of mean demand)")

## 6. LightGBM

In [ ]:
import lightgbm as lgb

gb = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1)
gb.fit(X_train, y_train)
pred_gb = np.clip(gb.predict(X_test), 0, None)
mae_gb = mean_absolute_error(y_test, pred_gb)
print(f"LightGBM MAE: {mae_gb:.2f}  ({mae_gb / y_test.mean() * 100:.1f}% of mean demand)")

## 7. Diagnose: why doesn't LightGBM win?

Compare train vs test error for both models. This rules out classic overfitting
(which would show very low train error and high test error).

In [ ]:
for name, mdl in [("Linear", lr), ("LightGBM", gb)]:
    tr = mean_absolute_error(y_train, np.clip(mdl.predict(X_train), 0, None))
    te = mean_absolute_error(y_test,  np.clip(mdl.predict(X_test),  0, None))
    print(f"{name:9s}  train {tr:5.2f}  |  test {te:5.2f}")

In [ ]:
# Which features the tree model leaned on most
imp = pd.Series(gb.feature_importances_, index=feat_cols).sort_values(ascending=False)
print(imp)

## 8. Summary

**Setup.** Predict daily trip demand per H3 cell for the 50 busiest cells. Target = trips
per (cell, day), missing days filled with 0. Features: calendar, cell identity and
baseline level, and past demand (lags + rolling means). Split by time: train 2022-2023,
test 2024.

**Result.** Linear regression reaches MAE ~7.8 (about 23% of mean demand). LightGBM
(MAE ~10.2) does **not** beat it.

**Why LightGBM doesn't win (checked, not a bug).** Features are clean (no NaN), and
LightGBM is not classically overfitting (its train error is not near zero). What happens:
LightGBM fits 2022-2023 more tightly, but some of that structure is specific to those
years and does not hold in 2024 (the market got more stable post-pandemic — both models
actually score *better* on 2024 than on the training years). The predictable signal here
is mostly a stable, near-linear pattern ("today is close to this cell's recent average"),
which the linear model captures and generalizes well. So for this task the simple model
is the right choice.

**Limits.** Daily, top-50-cell scope (the sample can't support hourly or rare cells).
Demand = completed trips (a proxy; unmet requests are not in the data). No weather or
event features yet — likely the biggest remaining lever to lower error.